# Stive (Stack)

Stiva este o structură **LIFO** (*Last In, First Out*): ultimul element introdus este primul scos.

Operații, toate în **O(1)**:
- `push(x)` – adaugă în vârf
- `pop()` – scoate și întoarce vârful
- `top()` / `peek()` – întoarce vârful fără să-l scoată
- `empty()` – verifică dacă stiva e goală

## 1. Stivă implementată cu vector (listă Python)

In [ ]:
class Stack:
    def __init__(self):
        self._data = []

    def push(self, x):
        self._data.append(x)

    def pop(self):
        if self.empty():
            raise IndexError("pop din stivă goală")
        return self._data.pop()

    def top(self):
        if self.empty():
            raise IndexError("top pe stivă goală")
        return self._data[-1]

    def empty(self):
        return len(self._data) == 0

    def __len__(self):
        return len(self._data)

    def __repr__(self):
        return f"Stack(baza -> vârf: {self._data})"


s = Stack()
for x in [1, 2, 3]:
    s.push(x)
print(s)
print("pop:", s.pop(), "| top:", s.top(), "|", s)

## 2. Stivă implementată cu listă înlănțuită

Vârful stivei = capul listei. `push` = inserare la început, `pop` = ștergere de la început.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value = value
        self.next = next


class LinkedStack:
    def __init__(self):
        self.top_node = None
        self.size = 0

    def push(self, x):
        self.top_node = Node(x, self.top_node)
        self.size += 1

    def pop(self):
        if self.top_node is None:
            raise IndexError("pop din stivă goală")
        x = self.top_node.value
        self.top_node = self.top_node.next
        self.size -= 1
        return x

    def top(self):
        if self.top_node is None:
            raise IndexError("top pe stivă goală")
        return self.top_node.value

    def empty(self):
        return self.top_node is None


s = LinkedStack()
for c in "abc":
    s.push(c)
print([s.pop() for _ in range(3)])   # ordine inversă

## 3. Aplicație: paranteze echilibrate

Parcurgem șirul: la o paranteză deschisă facem `push`; la una închisă verificăm că vârful este perechea ei și facem `pop`. La final stiva trebuie să fie goală.

In [ ]:
def is_balanced(text):
    pairs = {")": "(", "]": "[", "}": "{"}
    s = Stack()
    for c in text:
        if c in "([{":
            s.push(c)
        elif c in ")]}":
            if s.empty() or s.pop() != pairs[c]:
                return False
    return s.empty()


for t in ["([]{()})", "([)]", "((", "", "{[()()]}"]:
    print(f"{t!r:12} -> {is_balanced(t)}")

## 4. Aplicație: evaluarea unei expresii în formă postfixată

Forma postfixată (poloneză inversă): `3 4 + 2 *` = `(3 + 4) * 2`.

Operanzii se pun pe stivă; la un operator scoatem doi operanzi, calculăm și punem rezultatul înapoi.

In [ ]:
def eval_postfix(expr):
    s = Stack()
    for token in expr.split():
        if token in "+-*/":
            b = s.pop()      # atenție la ordine!
            a = s.pop()
            if token == "+": s.push(a + b)
            elif token == "-": s.push(a - b)
            elif token == "*": s.push(a * b)
            else: s.push(a / b)
        else:
            s.push(float(token))
    return s.pop()


print(eval_postfix("3 4 + 2 *"))          # 14
print(eval_postfix("5 1 2 + 4 * + 3 -"))  # 14

## 5. Aplicație: transformare infix → postfix (algoritmul Shunting-yard)

- operand → direct în ieșire
- `(` → pe stivă
- `)` → scoatem din stivă în ieșire până la `(`
- operator → scoatem din stivă operatorii cu prioritate ≥ și apoi îl punem pe stivă

In [ ]:
def infix_to_postfix(expr):
    precedence = {"+": 1, "-": 1, "*": 2, "/": 2}
    output, s = [], Stack()
    for token in expr.replace("(", " ( ").replace(")", " ) ").split():
        if token == "(":
            s.push(token)
        elif token == ")":
            while s.top() != "(":
                output.append(s.pop())
            s.pop()
        elif token in precedence:
            while not s.empty() and s.top() != "(" and precedence[s.top()] >= precedence[token]:
                output.append(s.pop())
            s.push(token)
        else:
            output.append(token)
    while not s.empty():
        output.append(s.pop())
    return " ".join(output)


expr = "(3 + 4) * 2 - 10 / 5"
postfix = infix_to_postfix(expr)
print(expr, "->", postfix, "=", eval_postfix(postfix))